# Quantum Machine Learning for Option Pricing — Distributed Experiment (Dask CPU)

## Overview

This notebook replicates the baseline experiment from `01_supervised_vs_unsupervised_qml_option_pricing.ipynb` with a key difference: **loss and gradient evaluations are parallelised across a local Dask cluster** instead of running sequentially on a single CPU thread.

The goal is to evaluate whether Dask-based parallelism reduces wall-clock training time for the same supervised/unsupervised QML experiment, without changing the model architecture, hyperparameters, or pricing methodology.

---

## Problem Setup

Identical to the baseline notebook:
- **Asset model:** Black-Scholes log-normal with $S_0=100$, $r=10\%$, $\sigma=25\%$, $T=1$ year
- **Strikes:** $K \in \{90, 100, 110\}$ (OTM, ATM, ITM)
- **Input feature:** log-moneyness $x_t = \log(S_T / K)$, rescaled to $[-\pi, \pi]$

---

## Dask Cluster Configuration

A `LocalCluster` is created at notebook startup with:
- **Workers:** 4 independent processes
- **Threads per worker:** 1 (avoids Python GIL contention)
- **Memory limit:** auto-managed by Dask
- **Dashboard:** disabled

The `dask_client` handle is passed into the `workflow_cfg` dictionary and forwarded to `adam_optimizer_loop`, which distributes individual loss/gradient evaluations across workers. If Dask is unavailable, the notebook falls back to sequential execution.

Resources are explicitly closed at the end of the notebook via `client.close()` and `cluster.close()`.

---

## Quantum Circuit Architecture

Identical to the baseline experiment:
- **Hardware-efficient ansatz** via `qml4var.architectures.hardware_efficient_ansatz`
- **Supervised mode:** 3 qubits/feature × 3 layers
- **Unsupervised mode:** 2 qubits/feature × 2 layers
- Circuit output interpreted as CDF; derivative gives PDF
- Gradients via **PyTorch autograd**

---

## Training Modes

### Supervised
- **Target:** Empirical CDF shifted to $[-0.5, 0.5]$
- **Loss:** MSE (0.9) + regularisation (0.1)
- **Learning rate:** 0.005 | **Dataset sizes:** 250, 500, 1000

### Unsupervised
- **Target:** No labels — distributional loss on PDF
- **Loss weights:** [0.2 PDF, 0.8 auxiliary]
- **Learning rate:** 0.1 | **Dataset sizes:** 1000, 2500, 5000

**Common settings:** Adam ($\beta_1=0.9$, $\beta_2=0.999$), early stopping (patience = 25, tol = $10^{-8}$), max 50 epochs.

These configurations are identical to the baseline, enabling a direct performance comparison.

---

## Option Pricing Method

Both supervised and unsupervised modes use **Method I — PDF-based Fourier pricing** exclusively:

1. Evaluate the trained circuit PDF on a 1024-point grid over $[-2\pi, 2\pi]$
2. Normalise the PDF to unit area
3. Compute Fourier coefficients $(A_k, B_k)$ of the PDF ($K=12$ terms)
4. Compute Fourier coefficients of the put payoff $h(x) = \max(K(1 - e^x), 0)$
5. Apply the pricing formula via `finance.fourier_price_v_t0`

> **Note:** Unlike the baseline notebook, this notebook does **not** use the Integration by Parts (Method II) formula for the unsupervised mode. The focus here is on training throughput via parallelism, not on pricing accuracy comparison between methods.

---

## Evaluation

Same metrics as the baseline:
- **Test MSE:** CDF prediction error on held-out test points
- **Estimated price vs. Black-Scholes price**
- **Runtime:** Seconds per training run (the primary comparison axis against the sequential baseline)

Results are aggregated over repetitions and plotted as price curves vs. dataset size per strike.

---

## Execution

- **Device:** CPU (Dask worker processes do not share GPU memory)
- **Parallelism:** 4-worker Dask `LocalCluster`
- **Debug mode:** Set `DEBUG_FAST_RUN = True` for a fast 2-epoch sanity check

In [ ]:
import sys
from time import perf_counter

import torch

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from finance import (
    bs_put_price,
    estimate_price_from_trained_pqc,
)
from qml4var.adam import adam_optimizer_loop
from qml4var.architectures import (
    hardware_efficient_ansatz,
    init_weights,
    normalize_data,
)
from qml4var.data_utils import (
    empirical_cdf,
    simulate_black_scholes_data_rescaled,
)
from qml4var.losses import torch_gradient
from qml4var.workflows import (
    qdml_loss_workflow,
    unsupervised_qdml_loss_workflow,
    workflow_for_cdf,
)

try:
    from tqdm.auto import tqdm
    TQDM_AVAILABLE = True
except Exception:
    TQDM_AVAILABLE = False

    class _NoTqdm:
        def __init__(self, total=None, desc=None, unit=None, disable=False):
            self.total = total
            self.n = 0
            self.disable = disable
            if not self.disable:
                print(f"{desc or 'Progress'} | total={total} {unit or ''}")

        def update(self, n=1):
            self.n += n

        def set_postfix(self, *args, **kwargs):
            return None

        def close(self):
            return None

    def tqdm(*args, **kwargs):
        return _NoTqdm(*args, **kwargs)


## 1. Global Setup

In [ ]:
# Black-Scholes parameters
S0 = 100.0
r = 0.10
sigma = 0.25
T = 1.0
STRIKES = [90, 100, 110]

# Domains
TRAIN_INTERVAL = (-2.0 * np.pi, 2.0 * np.pi)  # workflow domain
DATA_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)   # data interval after rescaling
ENCODING_INTERVAL = (-1.0 * np.pi, 1.0 * np.pi)
FEATURES_NUMBER = 1

# Fourier pricing setup
K_TERMS = 12
GRID_POINTS_FOURIER = 1024

# Optional: use local Dask cluster (CPU parallelism)
USE_LOCAL_DASK = True
DASK_N_WORKERS = 4
DASK_THREADS_PER_WORKER = 1
DASK_MEMORY_LIMIT = "auto"

DASK_CLIENT = None
_dask_cluster = None
if USE_LOCAL_DASK:
    try:
        from distributed import Client, LocalCluster
        _dask_cluster = LocalCluster(
            n_workers=DASK_N_WORKERS,
            threads_per_worker=DASK_THREADS_PER_WORKER,
            processes=True,
            memory_limit=DASK_MEMORY_LIMIT,
            dashboard_address=None,
        )
        DASK_CLIENT = Client(_dask_cluster)
        print(f"DASK local cluster active | workers={DASK_N_WORKERS} threads/worker={DASK_THREADS_PER_WORKER}")
    except Exception as ex:
        print(f"DASK disabled (fallback to serial execution): {ex}")
        DASK_CLIENT = None
        _dask_cluster = None

# Torch device (GPU if available, else CPU)
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("TORCH_DEVICE:", TORCH_DEVICE)

# Normalization from training domain [-2pi, 2pi] into encoding [-pi, pi]
base_frecuency, shift_feature = normalize_data(
    [TRAIN_INTERVAL[0]] * FEATURES_NUMBER,
    [TRAIN_INTERVAL[1]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[0]] * FEATURES_NUMBER,
    [ENCODING_INTERVAL[1]] * FEATURES_NUMBER,
)

print("base_frecuency:", base_frecuency)
print("shift_feature:", shift_feature)


## 2. Independent Configurations by Mode

You can edit each mode independently, without touching training functions.


In [11]:
MODE_CONFIGS = {
    "supervised": {
        "learning_rate": 0.005,
        "epochs": 50,
        "loss_weights": [0.9, 0.1],
        "train_sizes": [250, 500, 1000],
        "test_points": 100,
        "repetitions": 1,
        "beta1": 0.9,
        "beta2": 0.999,
        "tolerance": 1.0e-8,
        "n_counts_tolerance": 25,
        "print_step": 50,
        "batch_size": None,
        "n_qubits_by_feature": 3,
        "n_layers": 3,
        "debug_price_stats": True,
        "show_epoch_progress": True,
        "epoch_progress_leave": False,
    },
    "unsupervised": {
        "learning_rate": 0.1,
        "epochs": 50,
        "loss_weights": [0.2, 0.8],
        "train_sizes": [1000, 2500, 5000],
        "test_points": 1000,
        "repetitions": 1,
        "beta1": 0.9,
        "beta2": 0.999,
        "tolerance": 1.0e-8,
        "n_counts_tolerance": 25,
        "print_step": 50,
        "batch_size": None,
        "empirical_shift": -0.5,
        "n_qubits_by_feature": 2,
        "n_layers": 2,
        "debug_price_stats": True,
        "show_epoch_progress": True,
        "epoch_progress_leave": False,
    },
}

MODE_CONFIGS

{'supervised': {'learning_rate': 0.005,
  'epochs': 50,
  'loss_weights': [0.9, 0.1],
  'train_sizes': [250, 500, 1000],
  'test_points': 100,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 50,
  'batch_size': None,
  'n_qubits_by_feature': 3,
  'n_layers': 3,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False},
 'unsupervised': {'learning_rate': 0.1,
  'epochs': 50,
  'loss_weights': [0.2, 0.8],
  'train_sizes': [1000, 2500, 5000],
  'test_points': 1000,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 50,
  'batch_size': None,
  'empirical_shift': -0.5,
  'n_qubits_by_feature': 2,
  'n_layers': 2,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False}}

## 3. Helper Functions

In [ ]:
def build_mode_artifacts(cfg):
    pqc_cfg = {
        "features_number": FEATURES_NUMBER,
        "n_qubits_by_feature": int(cfg["n_qubits_by_feature"]),
        "n_layers": int(cfg["n_layers"]),
        "base_frecuency": list(base_frecuency),
        "shift_feature": list(shift_feature),
    }
    circuit_fn, weights_names, features_names = hardware_efficient_ansatz(**pqc_cfg)

    workflow_cfg = {
        "circuit_fn": circuit_fn,
        "weights_names": weights_names,
        "features_names": features_names,
        "torch_device": TORCH_DEVICE,
        "minval": [TRAIN_INTERVAL[0]],
        "maxval": [TRAIN_INTERVAL[1]],
        "points": 150,
    }
    return {
        "pqc_cfg": pqc_cfg,
        "weights_names": weights_names,
        "workflow_cfg": workflow_cfg,
    }


def batch_generator(X, Y, batch_size):
    return [(X[i : i + batch_size], Y[i : i + batch_size]) for i in range(0, len(X), batch_size)]


def run_supervised(u_train, cfg, artifacts, progress_desc=None):
    workflow_cfg = artifacts["workflow_cfg"]
    weights_names = artifacts["weights_names"]

    y_train = empirical_cdf(u_train).reshape((-1, 1)) - 0.5
    batch_size = cfg["batch_size"] if cfg["batch_size"] is not None else len(u_train)
    batches = batch_generator(u_train, y_train, batch_size)

    def training_loss(w_):
        return qdml_loss_workflow(
            w_, u_train, y_train, dask_client=DASK_CLIENT, loss_weights=cfg["loss_weights"], **workflow_cfg
        )

    def loss_for_grad(w_, x_, y_):
        return qdml_loss_workflow(
            w_, x_, y_, dask_client=DASK_CLIENT, loss_weights=cfg["loss_weights"], **workflow_cfg
        )

    def gradient_fn(w_, x_, y_):
        return torch_gradient(w_, x_, y_, loss_for_grad)

    optimizer_cfg = {
        "epochs": cfg["epochs"],
        "learning_rate": cfg["learning_rate"],
        "beta1": cfg["beta1"],
        "beta2": cfg["beta2"],
        "tolerance": cfg["tolerance"],
        "n_counts_tolerance": cfg["n_counts_tolerance"],
        "print_step": cfg["print_step"],
        "progress_bar": bool(cfg.get("show_epoch_progress", False)),
        "progress_desc": progress_desc if progress_desc is not None else "supervised epochs",
        "progress_leave": bool(cfg.get("epoch_progress_leave", False)),
    }

    initial_weights = init_weights(weights_names)
    return adam_optimizer_loop(
        weights_dict=initial_weights,
        loss_function=training_loss,
        metric_function=None,
        gradient_function=gradient_fn,
        batch_generator=batches,
        initial_time=0,
        **optimizer_cfg,
    )


def run_unsupervised(u_train, cfg, artifacts, progress_desc=None):
    workflow_cfg = artifacts["workflow_cfg"]
    weights_names = artifacts["weights_names"]

    dummy_y = np.zeros((u_train.shape[0], 1))
    batch_size = cfg["batch_size"] if cfg["batch_size"] is not None else len(u_train)
    batches = batch_generator(u_train, dummy_y, batch_size)

    def training_loss(w_):
        return unsupervised_qdml_loss_workflow(
            w_,
            u_train,
            dask_client=DASK_CLIENT,
            empirical_shift=cfg["empirical_shift"],
            loss_weights=cfg["loss_weights"],
            **workflow_cfg,
        )

    def loss_for_grad(w_, x_, y_):
        return unsupervised_qdml_loss_workflow(
            w_,
            x_,
            dask_client=DASK_CLIENT,
            empirical_shift=cfg["empirical_shift"],
            loss_weights=cfg["loss_weights"],
            **workflow_cfg,
        )

    def gradient_fn(w_, x_, y_):
        return torch_gradient(w_, x_, y_, loss_for_grad)

    optimizer_cfg = {
        "epochs": cfg["epochs"],
        "learning_rate": cfg["learning_rate"],
        "beta1": cfg["beta1"],
        "beta2": cfg["beta2"],
        "tolerance": cfg["tolerance"],
        "n_counts_tolerance": cfg["n_counts_tolerance"],
        "print_step": cfg["print_step"],
        "progress_bar": bool(cfg.get("show_epoch_progress", False)),
        "progress_desc": progress_desc if progress_desc is not None else "unsupervised epochs",
        "progress_leave": bool(cfg.get("epoch_progress_leave", False)),
    }

    initial_weights = init_weights(weights_names)
    return adam_optimizer_loop(
        weights_dict=initial_weights,
        loss_function=training_loss,
        metric_function=None,
        gradient_function=gradient_fn,
        batch_generator=batches,
        initial_time=0,
        **optimizer_cfg,
    )


## 4. Debug

Use these toggles before launching the full loop.


In [13]:
# Optional quick sanity mode
DEBUG_FAST_RUN = False

# More frequent progress logs (helps confirm the process is alive)
VERBOSE_PROGRESS = True

if VERBOSE_PROGRESS:
    for cfg in MODE_CONFIGS.values():
        cfg["print_step"] = min(int(cfg.get("print_step", 50)), 10)
        cfg["print_step"] = max(cfg["print_step"], 1)

if DEBUG_FAST_RUN:
    # tiny run to verify everything end-to-end before expensive execution
    for cfg in MODE_CONFIGS.values():
        cfg["epochs"] = min(int(cfg["epochs"]), 2)
        cfg["repetitions"] = 1
        cfg["train_sizes"] = [int(cfg["train_sizes"][0])]

MODE_CONFIGS


{'supervised': {'learning_rate': 0.005,
  'epochs': 50,
  'loss_weights': [0.9, 0.1],
  'train_sizes': [250, 500, 1000],
  'test_points': 100,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 10,
  'batch_size': None,
  'n_qubits_by_feature': 3,
  'n_layers': 3,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False},
 'unsupervised': {'learning_rate': 0.1,
  'epochs': 50,
  'loss_weights': [0.2, 0.8],
  'train_sizes': [1000, 2500, 5000],
  'test_points': 1000,
  'repetitions': 1,
  'beta1': 0.9,
  'beta2': 0.999,
  'tolerance': 1e-08,
  'n_counts_tolerance': 25,
  'print_step': 10,
  'batch_size': None,
  'empirical_shift': -0.5,
  'n_qubits_by_feature': 2,
  'n_layers': 2,
  'debug_price_stats': True,
  'show_epoch_progress': True,
  'epoch_progress_leave': False}}

In [14]:
def expected_total_runs(mode_configs, strikes):
    return sum(len(cfg["train_sizes"]) * len(strikes) * int(cfg["repetitions"]) for cfg in mode_configs.values())


def format_eta(seconds):
    if seconds is None or (not np.isfinite(seconds)):
        return "n/a"
    seconds = int(max(0, round(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"


def eta_from_partial(results_df, mode_configs, strikes):
    total_runs = expected_total_runs(mode_configs, strikes)
    done_runs = len(results_df)
    if done_runs == 0:
        return {"total_runs": total_runs, "done_runs": 0, "avg_sec_per_run": None, "eta_hours": None}
    avg_sec = float(results_df["seconds"].mean())
    pending = max(total_runs - done_runs, 0)
    eta_hours = (pending * avg_sec) / 3600.0
    return {
        "total_runs": total_runs,
        "done_runs": done_runs,
        "avg_sec_per_run": avg_sec,
        "eta_hours": eta_hours,
    }


USE_TQDM = True

# Usage after (or during) the main loop:
# eta_from_partial(results_df, MODE_CONFIGS, STRIKES)


## 4. Run Experiment 

In [15]:
import os

os.environ["QML4VAR_PROFILE_TRAINING"] = "1"
os.environ["QML4VAR_PROFILE_EPOCHS"] = "1"
os.environ["QML4VAR_PROFILE_ONCE"] = "1"
os.environ["QML4VAR_PROFILE_LABEL"] = "first_case"

os.environ["QML4VAR_PROFILE_GRAD"] = "1"
os.environ["QML4VAR_PROFILE_GRAD_FIRST_ONLY"] = "1"


In [16]:
results = []
base_seed = 2026

mode_artifacts = {}
for mode_name, cfg in MODE_CONFIGS.items():
    artifacts = build_mode_artifacts(cfg)
    mode_artifacts[mode_name] = artifacts
    print(
        f"mode={mode_name} -> qubits_by_feature={cfg['n_qubits_by_feature']} "
        f"layers={cfg['n_layers']} total_weights={len(artifacts['weights_names'])}"
    )

total_runs = expected_total_runs(MODE_CONFIGS, STRIKES)
print(f"\nPlanned training runs: {total_runs}")

progress = tqdm(
    total=total_runs,
    desc="Training runs",
    unit="run",
    disable=not (USE_TQDM and TQDM_AVAILABLE),
)

run_counter = 0
global_t0 = perf_counter()

for mode_name, cfg in MODE_CONFIGS.items():
    artifacts = mode_artifacts[mode_name]
    print(f"\n=== Running mode: {mode_name} ===")
    for K_ in STRIKES:
        theor_price = bs_put_price(S0, K_, r, sigma, T)
        for n_train in cfg["train_sizes"]:
            n_test = int(cfg["test_points"])
            for rep in range(cfg["repetitions"]):
                seed = base_seed + 100000 * rep + 1000 * int(K_) + n_train
                n_total = n_train + n_test

                x_raw_all, u_all, x_min_raw, x_max_raw = simulate_black_scholes_data_rescaled(
                    S0_=S0, r_=r, T_=T, sigma_=sigma, K_=K_, n_points=n_total, seed=seed
                )
                u_train = u_all[:n_train]
                u_test = u_all[n_train:]

                t0 = perf_counter()
                epoch_desc = f"{mode_name} K={K_} n={n_train} rep={rep + 1}"
                if mode_name == "supervised":
                    weights = run_supervised(u_train, cfg, artifacts, progress_desc=epoch_desc)
                else:
                    weights = run_unsupervised(u_train, cfg, artifacts, progress_desc=epoch_desc)
                t1 = perf_counter()

                # Optional test metric (CDF MSE wrt empirical CDF on test split)
                y_test = empirical_cdf(u_test).reshape((-1, 1)) - 0.5
                y_pred_test = workflow_for_cdf(
                    weights, u_test, dask_client=DASK_CLIENT, **artifacts["workflow_cfg"]
                )["y_predict_cdf"].reshape((-1, 1))
                test_mse = float(np.mean((y_pred_test - y_test) ** 2))

                debug_label = f"mode={mode_name} K={K_} n_train={n_train} rep={rep + 1}"
                est_price = estimate_price_from_trained_pqc(
                    weights=weights,
                    artifacts=artifacts,
                    K_=K_,
                    x_min_raw=x_min_raw,
                    x_max_raw=x_max_raw,
                    train_interval=TRAIN_INTERVAL,
                    risk_free_rate=r,
                    delta_t=T,
                    k_terms=K_TERMS,
                    grid_points=GRID_POINTS_FOURIER,
                    dask_client=DASK_CLIENT,
                    debug=bool(cfg.get("debug_price_stats", False)),
                    debug_label=debug_label,
                )

                run_seconds = float(t1 - t0)
                results.append({
                    "mode": mode_name,
                    "strike": K_,
                    "dataset_size": n_train,
                    "test_points": n_test,
                    "rep": rep + 1,
                    "estimated_price": est_price,
                    "theoretical_price": float(theor_price),
                    "test_mse": test_mse,
                    "seconds": run_seconds,
                    "n_qubits_by_feature": int(cfg["n_qubits_by_feature"]),
                    "n_layers": int(cfg["n_layers"]),
                })

                run_counter += 1
                elapsed = perf_counter() - global_t0
                avg_per_run = elapsed / max(run_counter, 1)
                eta_seconds = (total_runs - run_counter) * avg_per_run

                progress.update(1)
                progress.set_postfix({
                    "mode": mode_name,
                    "K": K_,
                    "n": n_train,
                    "rep": rep + 1,
                    "run_s": f"{run_seconds:.1f}",
                    "eta": format_eta(eta_seconds),
                })

                print(
                    f"mode={mode_name} K={K_} n_train={n_train} rep={rep + 1} "
                    f"price={est_price:.6f} run_s={run_seconds:.2f} ETA={format_eta(eta_seconds)}"
                )

progress.close()

total_elapsed = perf_counter() - global_t0
print(f"\nCompleted {run_counter}/{total_runs} runs in {format_eta(total_elapsed)}")

results_df = pd.DataFrame(results)
results_df.head()


mode=supervised -> qubits_by_feature=3 layers=3 total_weights=9
mode=unsupervised -> qubits_by_feature=2 layers=2 total_weights=4

Planned training runs: 18



Training runs:   0%|          | 0/18 [05:58<?, ?run/s]



=== Running mode: supervised ===


supervised K=90 n=250 rep=1:   0%|          | 0/50 [00:00<?, ?epoch/s]

[PROFILE GRAD] weights=9 loss_evals=18 eval_mean_s=12.540 eval_max_s=16.638 total_grad_s=225.719


...

	 MSE at t=40: None
	 Iteracion: 40. Loss: -0.029644148568517124


Maximum number of iterations achieved.




Training runs: 100%|██████████| 18/18 [191:33:35<00:00, 38311.99s/run, mode=unsupervised, K=110, n=5000, rep=1, run_s=24642.5, eta=00:00:00]

[PRICE DEBUG] mode=unsupervised K=110 n_train=5000 rep=1 pdf_min=0.000000e+00 pdf_max=8.843955e-02 area=3.539901e-01
mode=unsupervised K=110 n_train=5000 rep=1 price=20.028955 run_s=24642.45 ETA=00:00:00

Completed 18/18 runs in 92:57:39


,mode,strike,dataset_size,test_points,rep,estimated_price,theoretical_price,test_mse,seconds,n_qubits_by_feature,n_layers
0,supervised,90,250,100,1,4.361197,2.598827,0.020666,13209.840609,3,3
1,supervised,90,500,100,1,1.149780,2.598827,0.070541,22134.616456,3,3
2,supervised,90,1000,100,1,1.734323,2.598827,0.011726,37153.930585,3,3
3,supervised,100,250,100,1,2.359303,5.459533,0.043246,12230.537134,3,3
4,supervised,100,500,100,1,2.490515,5.459533,0.015662,21970.750438,3,3


## 5. Summaries

In [17]:
summary_df = results_df.groupby(["mode", "strike", "dataset_size"], as_index=False).agg(
    mean_price=("estimated_price", "mean"),
    std_price=("estimated_price", "std"),
    mean_test_mse=("test_mse", "mean"),
    mean_seconds=("seconds", "mean"),
)
summary_df


,mode,strike,dataset_size,mean_price,std_price,mean_test_mse,mean_seconds
0,supervised,90,250,4.361197,NaN,0.020666,13209.840609
1,supervised,90,500,1.149780,NaN,0.070541,22134.616456
2,supervised,90,1000,1.734323,NaN,0.011726,37153.930585
3,supervised,100,250,2.359303,NaN,0.043246,12230.537134
4,supervised,100,500,2.490515,NaN,0.015662,21970.750438
5,supervised,100,1000,5.422773,NaN,0.024551,34645.353207
6,supervised,110,250,6.709539,NaN,0.017790,11648.625018
7,supervised,110,500,13.247052,NaN,0.007892,18098.225332
8,supervised,110,1000,12.785986,NaN,0.031156,30986.422247
9,unsupervised,90,1000,9.667644,NaN,0.046187,5047.110141


## 6. Price Curves by Dataset Size


In [ ]:
colors = ["#377eb8", "#ff7f00", "#4daf4a"]

for mode_name, cfg in MODE_CONFIGS.items():
    dataset_sizes = cfg["train_sizes"]
    df_mode = results_df[results_df["mode"] == mode_name].copy()

    plt.figure(figsize=(10, 7))

    for K_, color in zip(STRIKES, colors, strict=False):
        df_k = df_mode[df_mode["strike"] == K_]

        means = []
        stds = []
        for n_ in dataset_sizes:
            vals = df_k[df_k["dataset_size"] == n_]["estimated_price"].values
            means.append(np.nanmean(vals) if vals.size > 0 else np.nan)
            stds.append(np.nanstd(vals, ddof=1) if vals.size > 1 else 0.0)

        plt.errorbar(
            dataset_sizes,
            means,
            yerr=stds,
            fmt="-o",
            color=color,
            label=f"K = {K_}",
            capsize=5,
            markersize=8,
            elinewidth=1.5,
        )

        # Theoretical line for each strike
        p_theo = bs_put_price(S0, K_, r, sigma, T)
        plt.axhline(p_theo, color=color, linestyle="--", linewidth=0.9, alpha=0.9)

        # Outliers by IQR
        for n_ in dataset_sizes:
            vals = df_k[df_k["dataset_size"] == n_]["estimated_price"].dropna().values
            if vals.size < 4:
                continue
            q1, q3 = np.percentile(vals, [25, 75])
            iqr = q3 - q1
            low = q1 - 1.5 * iqr
            high = q3 + 1.5 * iqr
            outliers = vals[(vals < low) | (vals > high)]
            if outliers.size > 0:
                plt.scatter(
                    np.full(outliers.shape[0], n_),
                    outliers,
                    marker="x",
                    color=color,
                    s=45,
                    alpha=0.95,
                    label=f"Outliers K={K_}" if n_ == dataset_sizes[0] else None,
                )

    plt.title(f"Estimated prices by dataset size | mode={mode_name} | test_points={cfg['test_points']}")
    plt.xlabel("Size of the dataset (n_train)")
    plt.ylabel("Price")
    plt.xticks(dataset_sizes)
    plt.grid(alpha=0.25, axis="y")
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()


## 7. Optional Dask Cleanup


In [19]:
if DASK_CLIENT is not None:
    DASK_CLIENT.close()
if _dask_cluster is not None:
    _dask_cluster.close()
print("DASK resources closed")


DASK resources closed
